# Fruit Classification — 02 · Preprocessing: a leakage-aware split

In [01_eda](01_eda.ipynb) we saw that the official Fruits-360 split **interleaves consecutive video frames** across Training/Validation/Test: frames `k`/`k+2` go to Training, `k+1` to Validation and `k+3` to Test. Since neighbouring frames of a rotating fruit are nearly identical, validation and test scores on that split are heavily inflated (**data leakage**).

This notebook builds a replacement split and writes it to **`data/splits.csv`**, which both modelling notebooks consume:

**Idea — contiguous-block split.** Each (class, rotation axis) pair is one video. Instead of interleaving its frames, we sort them by frame index and cut the video into three consecutive blocks:

- first ≈70% of frames → `train`
- next ≈15% → `val`
- last ≈15% → `test`

Now only the handful of frames right at the two block boundaries are temporal neighbours across sets — instead of *every* frame. The split is fully deterministic (no randomness) and **no files are moved**; the CSV simply maps each existing file to its new split.

*Remaining limitation (from 01_eda):* each class contains only one physical object, so the models are evaluated on unseen views of known fruits, not on unseen fruits. No split can fix that for this dataset.

## 1. Setup

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.isdir("cnn_vgg16__fruit_classification"):
        !git clone https://github.com/Adriana394/cnn_vgg16__fruit_classification.git
    REPO_DIR = "cnn_vgg16__fruit_classification"
else:
    REPO_DIR = ".."  # this notebook lives in <repo>/notebooks/

DATA_ROOT = os.path.join(REPO_DIR, "data")
IMAGES_DIR = os.path.join(DATA_ROOT, "Images")
SPLITS_CSV = os.path.join(DATA_ROOT, "splits.csv")
print("Images directory:", os.path.abspath(IMAGES_DIR))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

## 2. Index all images

We pool the images from all three official folders into one DataFrame. The original folder (`Training`/`Validation`/`Test`) is kept only as part of the file path — it plays no role in the new split.

`filepath` is stored relative to `data/` with forward slashes, so the CSV works on any operating system.

In [ ]:
records = []
for folder in ["Training", "Validation", "Test"]:
    folder_dir = os.path.join(IMAGES_DIR, folder)
    for label in sorted(os.listdir(folder_dir)):
        class_dir = os.path.join(folder_dir, label)
        if not os.path.isdir(class_dir):
            continue
        for filename in sorted(os.listdir(class_dir)):
            axis, frame = filename.rsplit(".", 1)[0].rsplit("_", 1)
            records.append(
                {
                    "filepath": f"Images/{folder}/{label}/{filename}",
                    "label": label,
                    "axis": axis,
                    "frame": int(frame),
                }
            )

images = pd.DataFrame(records)
print(f"Total images: {len(images)}")
print(f"Classes: {images['label'].nunique()}")
print(f"Videos (class × axis): {len(images.groupby(['label', 'axis']))}")

## 3. Contiguous-block split per video

For every (class, axis) video: sort frames by index, then assign the first 70% to `train`, the next 15% to `val` and the rest to `test`.

In [ ]:
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15  # the remaining ~15% become the test block

parts = []
for (label, axis), video in images.groupby(["label", "axis"]):
    video = video.sort_values("frame").reset_index(drop=True)
    n = len(video)
    n_train = round(n * TRAIN_FRAC)
    n_val = round(n * VAL_FRAC)
    assignment = ["train"] * n_train + ["val"] * n_val + ["test"] * (n - n_train - n_val)
    parts.append(video.assign(split=assignment))

splits = pd.concat(parts, ignore_index=True).sort_values("filepath").reset_index(drop=True)
splits["split"].value_counts()

### Old vs. new split on one video

The timeline below shows which set each frame of the `apple_6`/`r0` video belongs to — in the official split (top) and in our block split (bottom).

In [ ]:
video = splits[(splits["label"] == "apple_6") & (splits["axis"] == "r0")].sort_values("frame")
official = video["filepath"].str.split("/").str[1]  # original folder name

colors = {"Training": "tab:blue", "Validation": "tab:orange", "Test": "tab:green",
          "train": "tab:blue", "val": "tab:orange", "test": "tab:green"}

fig, axes = plt.subplots(2, 1, figsize=(14, 2.6), sharex=True)
axes[0].scatter(video["frame"], [0] * len(video), c=official.map(colors), marker="|", s=400)
axes[0].set_ylabel("official", rotation=0, ha="right", va="center")
axes[1].scatter(video["frame"], [0] * len(video), c=video["split"].map(colors), marker="|", s=400)
axes[1].set_ylabel("block split", rotation=0, ha="right", va="center")
for ax in axes:
    ax.set_yticks([])
axes[1].set_xlabel("frame index")
fig.suptitle("apple_6 / r0 — blue = train, orange = val, green = test", y=1.05)
plt.tight_layout()
plt.show()

In the official split (top) the three sets alternate frame by frame across the whole video. In the block split (bottom) each set is one contiguous chunk — only the two boundaries still have neighbouring frames in different sets.

## 4. Sanity checks

In [ ]:
assert len(splits) == len(images), "every image must be assigned exactly once"
assert not splits["filepath"].duplicated().any(), "no file may appear twice"
assert splits.groupby("label")["split"].nunique().eq(3).all(), "every class needs all three splits"

missing = [p for p in splits["filepath"] if not os.path.isfile(os.path.join(DATA_ROOT, p))]
assert not missing, f"{len(missing)} paths do not exist"

per_class = (
    splits.pivot_table(index="label", columns="split", values="filepath", aggfunc="count")
    .reindex(columns=["train", "val", "test"])
)
per_class.assign(total=per_class.sum(axis=1))

## 5. Save `data/splits.csv`

The CSV stores only what the modelling notebooks need: `filepath`, `label`, `split`. Because the whole procedure is deterministic, re-running this notebook always reproduces the identical file (it is also committed to the repository, so notebooks 03/04 work without running this one first).

In [ ]:
splits[["filepath", "label", "split"]].to_csv(SPLITS_CSV, index=False)
print(f"Wrote {len(splits)} rows to {os.path.abspath(SPLITS_CSV)}")
pd.read_csv(SPLITS_CSV).head()

## 6. What the modelling notebooks do with this

Both modelling notebooks load `data/splits.csv` and build their input pipelines from it:

- **03_modelling_keras** uses `ImageDataGenerator.flow_from_dataframe` with `directory=data/`.
- **04_modelling_pytorch** uses a small custom `Dataset` that reads the same CSV.

Shared preprocessing decisions (motivated in 01_eda):

- resize every image to **100×100** (sizes vary in this dataset version),
- **augmentation only on the training set** — validation and test stay untouched apart from resizing and normalisation,
- normalisation as each framework's pretrained VGG16 expects it (`preprocess_input` in Keras, ImageNet mean/std in PyTorch).